# Functions


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# DATA LOADING
current_dir = Path.cwd()
project_root = current_dir.parents[2]
path_data = project_root / 'DATA_PPMI/Results/MODEL_DATA/DATA_BL_V08.csv'
data = pd.read_csv(path_data,index_col=0)

# FUNCIONES PARA CALCULAR DELTA Y CLASIFICACIÓN HY3

def compute_delta(df_prev, df_next,sufix):

    # Alinear por index (inner join automático)
    df_prev, df_next = df_prev.align(df_next, join="inner", axis=0)
    
    # Seleccionar solo columnas numéricas
    numeric_cols = df_prev.select_dtypes(include='number').columns
    
    # Calcular delta
    delta = df_next[numeric_cols] - df_prev[numeric_cols]
    
    # Renombrar columnas
    delta.columns = [f"{col}_delta_{sufix}" for col in numeric_cols]
    
    return delta

def HY3_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    elif nhy  in [1,2]:
        return 1  # Early Stage
    else:
        return 2  # Advanced Stage
    
def HY4_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    elif nhy  == 1:
        return 1  # Early Stage
    elif nhy == 2:
        return 2  # Mid Stage
    else:
        return 3  # Advanced Stage

def HY43_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    elif nhy == 1:
        return 1  # Early Stage
    else:
        return 2  # Advanced Stage
    
def HY2_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    else:
        return 1  # Advanced Stage

def classification_mcid(x):
    if x <= -3:
        return 0  # Clinically meaningful improvement
    elif x >= 4:
        return 2   # Clinically meaningful worsening
    else:
        return 1   # Stable / no clinically meaningful change
    

# groupby para calcular estadísticas por paciente y visita

def compute_group_stats(df, group_level="PATNO", EVENT_ID:list=None):
    df = df.copy()
    df = df.loc[df["EVENT_ID"].isin(EVENT_ID),:]
    df.drop(columns=["EVENT_ID"], inplace=True)
    
    df_grouped = df.groupby(level=group_level)
    df_mean = df_grouped.mean().add_suffix("_mean")
    df_min  = df_grouped.min().add_suffix("_min")
    df_max  = df_grouped.max().add_suffix("_max")
    df_var  = df_grouped.var().add_suffix("_var")
    df_std  = df_grouped.std().add_suffix("_std")

    return pd.concat([df_mean, df_min, df_max, df_var, df_std], axis=1)
    



# V06 + Deltas

In [2]:
data_V06_delta = data.copy()


V08_cols = ['ENRLLRRK2', 'ENRLGBA', 'COHORT_DEFINITION', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT', "NHY"]
updrs3_cols=['NP3SPCH','NP3FACXP','NP3RIGN','NP3RIGRU','NP3RIGLU','NP3RIGRL','NP3RIGLL','NP3FTAPR','NP3FTAPL','NP3HMOVR','NP3HMOVL','NP3PRSPR','NP3PRSPL','NP3TTAPR','NP3TTAPL','NP3LGAGR','NP3LGAGL','NP3RISNG','NP3GAIT','NP3FRZGT','NP3PSTBL','NP3POSTR','NP3BRADY','NP3PTRMR','NP3PTRML','NP3KTRMR','NP3KTRML','NP3RTARU','NP3RTALU','NP3RTARL','NP3RTALL','NP3RTALJ','NP3RTCON']

data_last = data_V06_delta[V08_cols+['EVENT_ID']+updrs3_cols].copy()
data_last['UPDRS3_total'] = data_last[updrs3_cols].sum(axis=1)
data_last.drop(columns=updrs3_cols, inplace=True)
data_V08 = data_last.loc[data_last["EVENT_ID"] == 'V08',:]  # New Visit
data_V06 = data_last.loc[data_last["EVENT_ID"] == 'V06',:].copy()  # last visit
data_V08 = data_V08.sort_index()
data_V06 = data_V06.sort_index()
data_V08['Delta_UPDRS3_V08_V06'] = data_V08['UPDRS3_total'] - data_V06['UPDRS3_total']

data_V08["STAGE_HY3"] = data_V08["NHY"].apply(HY3_classification)
data_V08["STAGE_HY2"] = data_V08["NHY"].apply(HY2_classification)
data_V08["STAGE_HY4"] = data_V08["NHY"].apply(HY4_classification)
data_V08["STAGE_HY43"] = data_V08["NHY"].apply(HY43_classification)
data_V08["STAGE_MCID"] = data_V08["Delta_UPDRS3_V08_V06"].apply(classification_mcid)

data_V08.drop(columns=["EVENT_ID",'COHORT_DEFINITION','NHY'], inplace=True) 
data_V06_delta.drop(columns=['ENRLLRRK2', 'ENRLGBA', 'COHORT_DEFINITION', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT', "NHY"], inplace=True)

# Generar los df para delta 
BL_data = data_V06_delta.loc[data_V06_delta["EVENT_ID"] == 'BL',:].copy()
V04_data = data_V06_delta.loc[data_V06_delta["EVENT_ID"] == 'V04',:].copy()
V06_data = data_V06_delta.loc[data_V06_delta["EVENT_ID"] == 'V06',:].copy()

delta_V04_BL = compute_delta(BL_data, V04_data, "V04_BL")
delta_V06_V04 = compute_delta(V04_data, V06_data, "V06_V04")
delta_V06_BL = compute_delta(BL_data, V06_data, "V06_BL")

# last visit hist 
data_v06=data_V06_delta.loc[data_V06_delta["EVENT_ID"] == 'V06',:].copy()
data_v06.columns = data_v06.columns + "_V06"
data_v06.drop(columns=["EVENT_ID_V06"], inplace=True)

# Unir los df de delta con el df de la última visita
data_V06_delta = data_V08.merge(data_v06, left_index=True, right_index=True, how='inner')
data_V06_delta = data_V06_delta.merge(delta_V04_BL, left_index=True, right_index=True, how='inner')
data_V06_delta = data_V06_delta.merge(delta_V06_V04, left_index=True, right_index=True, how='inner')
data_V06_delta = data_V06_delta.merge(delta_V06_BL, left_index=True, right_index=True, how='inner')

data_V06_delta.drop(columns=["STAGE_HY3",'STAGE_HY2', "STAGE_HY4", "STAGE_HY43", "STAGE_MCID", "Delta_UPDRS3_V08_V06", "UPDRS3_total"]).to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V06+Deltas.csv')
data_V06_delta["STAGE_HY3"].to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3.csv')
data_V06_delta["STAGE_HY2"].to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY2.csv')
data_V06_delta["STAGE_HY4"].to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4.csv')
data_V06_delta["STAGE_HY43"].to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY43.csv')
data_V06_delta["STAGE_MCID"].to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID.csv')
data_V06_delta['Delta_UPDRS3_V08_V06'].to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_delta_UPDRS3_V08_V06.csv')
data_V06_delta['UPDRS3_total'].to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_UPDRS3_total.csv')

data_V06_delta.head() #con stage, datos de la última visita y deltas entre visitas


,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,AGE_AT_VISIT,UPDRS3_total,Delta_UPDRS3_V08_V06,STAGE_HY3,STAGE_HY2,...,NP3PTRMR_delta_V06_BL,NP3PTRML_delta_V06_BL,NP3KTRMR_delta_V06_BL,NP3KTRML_delta_V06_BL,NP3RTARU_delta_V06_BL,NP3RTALU_delta_V06_BL,NP3RTARL_delta_V06_BL,NP3RTALL_delta_V06_BL,NP3RTALJ_delta_V06_BL,NP3RTCON_delta_V06_BL
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,0,0,0.0,1.0,16.0,59.7,46.0,3.0,1,1,...,1.0,1.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0,1.0
3018,0,0,0.0,1.0,16.0,63.6,38.0,3.0,1,1,...,3.0,0.0,2.0,0.0,1.0,0.0,-1.0,0.0,0.0,3.0
3020,0,0,0.0,1.0,15.0,77.0,45.0,9.0,2,1,...,-2.0,0.0,0.0,0.0,1.0,2.0,2.0,0.0,0.0,2.0
3028,0,0,1.0,1.0,22.0,78.8,36.0,-13.0,1,1,...,0.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,2.0
3051,0,0,1.0,1.0,18.0,74.7,22.0,3.0,1,1,...,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0,-1.0


# V06 + Stats(BL+V04)

In [5]:
data_V06_stats = data.copy()

V08_cols = ['ENRLLRRK2', 'ENRLGBA', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT']
data_V08 = data_V06_stats[V08_cols+['EVENT_ID']].copy()
data_V08 = data_V08.loc[data_V08["EVENT_ID"] == 'V08',:]  # New Visit
data_V08.drop(columns=["EVENT_ID"], inplace=True) 

data_V06_stats.drop(columns=['ENRLLRRK2', 'ENRLGBA', 'COHORT_DEFINITION', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT', "NHY"], inplace=True)

# Generar los df para estadísticas
stats= compute_group_stats(data_V06_stats, group_level="PATNO", EVENT_ID=['BL','V04'])

data_v06=data_V06_stats.loc[data_V06_stats["EVENT_ID"] == 'V06',:].copy()
data_v06 = data_v06.add_suffix("_V06")
data_v06.drop(columns=["EVENT_ID_V06"], inplace=True)

data_V06_stats = data_V08.merge(data_v06, left_index=True, right_index=True, how='inner')
data_V06_stats = data_V06_stats.merge(stats, left_index=True, right_index=True, how='inner')
data_V06_stats.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V06+STATS.csv')
data_V06_stats.head() #datos de la última visita y estadísticas entre visitas


,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,AGE_AT_VISIT,MCAALTTM_V06,MCACUBE_V06,MCACLCKC_V06,MCACLCKN_V06,...,NP3PTRMR_std,NP3PTRML_std,NP3KTRMR_std,NP3KTRML_std,NP3RTARU_std,NP3RTALU_std,NP3RTARL_std,NP3RTALL_std,NP3RTALJ_std,NP3RTCON_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,0,0,0.0,1.0,16.0,59.7,1.0,1.0,1.0,1.0,...,0.000000,0.000000,0.000000,0.000000,1.414214,0.000000,0.000000,0.0,0.0,0.707107
3018,0,0,0.0,1.0,16.0,63.6,1.0,1.0,1.0,1.0,...,1.414214,1.414214,1.414214,0.707107,0.707107,0.707107,0.707107,0.0,0.0,1.414214
3020,0,0,0.0,1.0,15.0,77.0,0.0,1.0,1.0,1.0,...,0.000000,0.000000,0.000000,0.000000,0.707107,0.000000,1.414214,0.0,0.0,0.000000
3028,0,0,1.0,1.0,22.0,78.8,1.0,1.0,1.0,1.0,...,0.000000,0.000000,0.000000,0.707107,0.000000,0.707107,0.000000,0.0,0.0,0.707107
3051,0,0,1.0,1.0,18.0,74.7,1.0,1.0,1.0,1.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.707107


# Stats(BL+V04+V06)

In [6]:
data_stats = data.copy()

V08_cols = ['ENRLLRRK2', 'ENRLGBA', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT']
data_V08 = data_stats[V08_cols+['EVENT_ID']].copy()
data_V08 = data_V08.loc[data_V08["EVENT_ID"] == 'V08',:]  # New Visit
data_V08.drop(columns=["EVENT_ID"], inplace=True) 

data_stats.drop(columns=['ENRLLRRK2', 'ENRLGBA', 'COHORT_DEFINITION', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT', "NHY"], inplace=True)

# Generar los df para estadísticas
stats= compute_group_stats(data_stats, group_level="PATNO", EVENT_ID=['BL','V04','V06'])


data_stats = data_V08.merge(stats, left_index=True, right_index=True, how='inner')
data_stats.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_STATS.csv')
data_stats.head() #datos de la última visita y estadísticas entre visitas

,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,AGE_AT_VISIT,MCAALTTM_mean,MCACUBE_mean,MCACLCKC_mean,MCACLCKN_mean,...,NP3PTRMR_std,NP3PTRML_std,NP3KTRMR_std,NP3KTRML_std,NP3RTARU_std,NP3RTALU_std,NP3RTARL_std,NP3RTALL_std,NP3RTALJ_std,NP3RTCON_std
PATNO,,,,,,,,,,,,,,,,,,,,,
3003,0,0,0.0,1.0,16.0,59.7,1.000000,1.000000,1.0,1.0,...,0.577350,0.577350,0.000000,0.57735,1.154701,0.000000,0.000000,0.0,0.0,0.577350
3018,0,0,0.0,1.0,16.0,63.6,1.000000,0.666667,1.0,1.0,...,1.527525,1.154701,1.154701,0.57735,0.577350,0.577350,0.577350,0.0,0.0,1.527525
3020,0,0,0.0,1.0,15.0,77.0,0.666667,0.666667,1.0,1.0,...,1.154701,0.000000,0.000000,0.00000,0.577350,1.154701,1.154701,0.0,0.0,1.154701
3028,0,0,1.0,1.0,22.0,78.8,1.000000,1.000000,1.0,1.0,...,0.000000,0.000000,0.000000,0.57735,0.000000,1.527525,0.000000,0.0,0.0,1.527525
3051,0,0,1.0,1.0,18.0,74.7,1.000000,1.000000,1.0,1.0,...,0.577350,0.000000,0.000000,0.00000,0.577350,0.000000,0.000000,0.0,0.0,0.577350
